# 18 - Peak Statistics by Product (Dr. Yow)

This notebook lets you inspect:
- Peak statistics by product (same product universe as egime_eda_final)
- Time-series charts with detected peaks highlighted
- Product descriptions by dataset (M5/Favorita/Amazon) with manual name overrides for reviewed examples

In [ ]:
from pathlib import Path
import gzip
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

BASE = Path('.').resolve()
if (BASE / 'notebooks').exists() and not (BASE / 'reports').exists():
    BASE = BASE.parent

PEAK_DIR = BASE / 'reports' / 'peak_eda_final'
PLOTS_DIR = PEAK_DIR / 'plots_series'
REGIME_SUMMARY_PATH = BASE / 'reports' / 'regime_eda_final' / 'regime_summary.csv'
PEAK_SUMMARY_PATH = PEAK_DIR / 'peak_stats_summary.csv'
DATA_M5 = BASE / 'data' / 'raw' / 'm5'
DATA_FAV = BASE / 'data' / 'raw' / 'favorita'

# Prefer meta folder if available, else use review folder already present in this repo.
AMAZON_RAW = BASE / 'data' / 'raw' / 'amazon_2023'
META_AMZ = AMAZON_RAW / 'meta_categories'
if not META_AMZ.exists():
    META_AMZ = AMAZON_RAW / 'review_categories'

print('BASE:', BASE)
print('Peak summary exists:', PEAK_SUMMARY_PATH.exists())
print('Regime summary exists:', REGIME_SUMMARY_PATH.exists())
print('Plots dir exists:', PLOTS_DIR.exists())
print('Amazon metadata/review dir exists:', META_AMZ.exists(), '|', META_AMZ)

In [ ]:
peak_df = pd.read_csv(PEAK_SUMMARY_PATH)
regime_df = pd.read_csv(REGIME_SUMMARY_PATH)
peak_df.head(3)

In [ ]:
def build_m5_meta(path_m5):
    sales_path = None
    for name in ['sales_train_validation.csv', 'sales_train_evaluation.csv']:
        p = path_m5 / name
        if p.exists():
            sales_path = p
            break
    if sales_path is None:
        return pd.DataFrame(columns=['series_id','product_desc'])

    usecols = ['id','item_id','dept_id','cat_id','store_id','state_id']
    m5 = pd.read_csv(sales_path, usecols=usecols)
    m5['series_id'] = m5['id']
    m5['product_desc'] = (
        'M5 item=' + m5['item_id'].astype(str)
        + ' | dept=' + m5['dept_id'].astype(str)
        + ' | category=' + m5['cat_id'].astype(str)
        + ' | store=' + m5['store_id'].astype(str)
        + ' | state=' + m5['state_id'].astype(str)
    )
    return m5[['series_id','product_desc']]


def build_favorita_meta(path_fav):
    items_path = path_fav / 'items.csv'
    if not items_path.exists():
        return pd.DataFrame(columns=['series_id','product_desc'])

    items = pd.read_csv(items_path, usecols=['item_nbr','family','class','perishable'])
    items['series_id'] = 'item_' + items['item_nbr'].astype(str)
    items['product_desc'] = (
        'Favorita item=' + items['item_nbr'].astype(str)
        + ' | family=' + items['family'].astype(str)
        + ' | class=' + items['class'].astype(str)
        + ' | perishable=' + items['perishable'].astype(str)
    )
    return items[['series_id','product_desc']]


def build_amazon_meta(path_meta, asins):
    meta_candidates = [
        path_meta / 'meta_Health_and_Household.jsonl.gz',
        path_meta / 'meta_Home_and_Kitchen.jsonl.gz',
        path_meta / 'meta_Cell_Phones_and_Accessories.jsonl.gz',
        path_meta / 'Health_and_Household.jsonl.gz',
        path_meta / 'Cell_Phones_and_Accessories.jsonl.gz',
    ]
    meta_path = next((p for p in meta_candidates if p.exists()), None)

    rows = []
    asin_set = set(asins)

    if meta_path is not None and 'meta_' in meta_path.name:
        with gzip.open(meta_path, 'rt', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    continue
                asin = obj.get('parent_asin') or obj.get('asin')
                if asin not in asin_set:
                    continue
                title = obj.get('title') or obj.get('name') or 'unknown_title'
                main_cat = obj.get('main_category') or obj.get('category') or 'unknown_category'
                rows.append({'series_id': asin, 'product_desc': f'Amazon asin={asin} | title={title} | category={main_cat}'})
                asin_set.remove(asin)
                if not asin_set:
                    break

    if asin_set:
        for asin in asin_set:
            rows.append({'series_id': asin, 'product_desc': f'Amazon asin={asin} | product_name=unknown | source=review_only'})

    return pd.DataFrame(rows)


# Manual English names for products discussed/reviewed in this project.
PRODUCT_NAME_OVERRIDES = {
    'B0C1G1BJ2B': 'Paper roll',
    'B072KDRZH5': 'Household product (exact title unavailable in local files)',
    'HOUSEHOLD_1_187_WI_1_validation': 'M5 household SKU HOUSEHOLD_1_187 in WI_1 store',
    'item_789224': 'Favorita cleaning item 789224',
}

m5_meta = build_m5_meta(DATA_M5)
fav_meta = build_favorita_meta(DATA_FAV)
amz_asins = peak_df.loc[peak_df['dataset'] == 'AMAZON_2023', 'series_id'].unique().tolist()
amz_meta = build_amazon_meta(META_AMZ, amz_asins)

meta_df = pd.concat([m5_meta, fav_meta, amz_meta], ignore_index=True).drop_duplicates('series_id')
meta_df['product_desc'] = meta_df.apply(
    lambda r: PRODUCT_NAME_OVERRIDES.get(r['series_id'], r['product_desc']), axis=1
)
meta_df.head(5)

In [ ]:
merged = peak_df.merge(
    regime_df[['dataset','series_id','regime_label','sbc_class','zero_rate','adi','cv2','transition_score']],
    on=['dataset','series_id'],
    how='left'
).merge(
    meta_df,
    on='series_id',
    how='left'
)

cols = [
    'dataset','series_id','product_desc','regime_label','sbc_class',
    'n_peaks','peaks_per_365','avg_peak_height','max_peak_height',
    'avg_interpeak_days','peak_irregularity_cv','zero_rate','adi','cv2','transition_score'
]
merged = merged[cols]
display(merged.head(10))

In [ ]:
summary = merged.groupby('dataset', as_index=False)[
    ['n_peaks','peaks_per_365','avg_peak_height','max_peak_height','avg_interpeak_days','peak_irregularity_cv']
].mean(numeric_only=True).round(3)
display(summary)

In [ ]:
for ds in ['M5_WALMART','FAVORITA','AMAZON_2023']:
    sub = merged[merged['dataset'] == ds].sort_values('n_peaks', ascending=False).head(8)
    display(Markdown(f"### {ds} - Top 8 by n_peaks"))
    display(sub[['series_id','product_desc','regime_label','n_peaks','peaks_per_365','avg_peak_height','avg_interpeak_days']])

In [ ]:
def show_product(series_id, dataset=None):
    if dataset is None:
        row = merged[merged['series_id'] == series_id].head(1)
    else:
        row = merged[(merged['series_id'] == series_id) & (merged['dataset'] == dataset)].head(1)

    if row.empty:
        print('Series not found:', series_id, dataset)
        return

    r = row.iloc[0]
    fname = f"{r['dataset']}__{r['series_id']}__peaks.png"
    img_path = PLOTS_DIR / fname

    md_txt = (
        f"### {r['dataset']} | {r['series_id']}\n"
        f"- **Product**: {r['product_desc']}\n"
        f"- **Regime**: {r['regime_label']} ({r['sbc_class']})\n"
        f"- **n_peaks**: {r['n_peaks']} | **peaks_per_365**: {r['peaks_per_365']:.2f}\n"
        f"- **avg_peak_height**: {r['avg_peak_height']:.3f} | **max_peak_height**: {r['max_peak_height']:.3f}\n"
        f"- **avg_interpeak_days**: {r['avg_interpeak_days']:.2f} | **peak_irregularity_cv**: {r['peak_irregularity_cv']:.3f}"
    )
    display(Markdown(md_txt))

    if img_path.exists():
        display(Image(filename=str(img_path)))
    else:
        print('Plot not found:', img_path)

In [ ]:
show_product('HOUSEHOLD_1_187_WI_1_validation', dataset='M5_WALMART')
show_product('item_789224', dataset='FAVORITA')
show_product('B072KDRZH5', dataset='AMAZON_2023')

In [ ]:
for ds in ['M5_WALMART','FAVORITA','AMAZON_2023']:
    display(Markdown(f"## {ds} - Sample Peak Charts"))
    for sid in merged[merged['dataset']==ds].head(3)['series_id'].tolist():
        show_product(sid, dataset=ds)